# Azure ML: Train and Deploy Iris Endpoint
Run these cells in an Azure ML Notebook (inside your AML workspace).

In [ ]:
%pip install -q azure-ai-ml azure-identity scikit-learn joblib

In [ ]:
import json
from datetime import datetime
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient, command, Input, Output
from azure.ai.ml.entities import (
    Model,
    Environment,
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    CodeConfiguration,
)

## Workspace Config
Fill these values from your Azure ML workspace.

In [ ]:
subscription_id = "<AZURE_SUBSCRIPTION_ID>"
resource_group = "<AZURE_RESOURCE_GROUP>"
workspace_name = "<AZURE_ML_WORKSPACE_NAME>"
compute_name = "cpu-cluster"

credential = DefaultAzureCredential()
ml_client = MLClient(credential, subscription_id, resource_group, workspace_name)

endpoint_name = f"iris-endpoint-{datetime.utcnow().strftime('%Y%m%d%H%M%S')}".lower()
print("Endpoint name:", endpoint_name)

## Create Training and Scoring Files

In [ ]:
project_dir = Path("azure_iris")
project_dir.mkdir(exist_ok=True)

train_code = '''
import argparse
import json
from pathlib import Path

import joblib
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

parser = argparse.ArgumentParser()
parser.add_argument("--model-dir", type=str, required=True)
args = parser.parse_args()

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=400, random_state=42)),
])

model.fit(X_train, y_train)
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f"Validation accuracy: {acc:.4f}")

out_dir = Path(args.model_dir)
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, out_dir / "model.joblib")
with open(out_dir / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(iris.target_names.tolist(), f)
'''

score_code = '''
import json
import os

import joblib
import numpy as np

model = None
class_names = None

def init():
    global model, class_names
    model_dir = os.getenv("AZUREML_MODEL_DIR")
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    with open(os.path.join(model_dir, "class_names.json"), "r", encoding="utf-8") as f:
        class_names = json.load(f)

def run(raw_data):
    payload = json.loads(raw_data)
    data = payload.get("data", payload)

    if isinstance(data, list) and data and all(isinstance(v, (int, float)) for v in data):
        arr = np.array([data], dtype=float)
    else:
        arr = np.array(data, dtype=float)

    preds = model.predict(arr)
    labels = [class_names[int(p)] for p in preds]

    return {"prediction": labels[0] if len(labels) == 1 else labels}
'''

conda_yaml = '''
name: iris-inference-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - numpy
      - scikit-learn
      - joblib
      - azureml-inference-server-http
'''

(project_dir / "train.py").write_text(train_code, encoding="utf-8")
(project_dir / "score.py").write_text(score_code, encoding="utf-8")
(project_dir / "conda.yml").write_text(conda_yaml, encoding="utf-8")

print("Created:", project_dir / "train.py")
print("Created:", project_dir / "score.py")
print("Created:", project_dir / "conda.yml")

## Submit Training Job

In [ ]:
train_job = command(
    code=str(project_dir),
    command="python train.py --model-dir ${{outputs.model_dir}}",
    outputs={"model_dir": Output(type="uri_folder")},
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute=compute_name,
    display_name="iris-train-job",
)

returned_job = ml_client.jobs.create_or_update(train_job)
ml_client.jobs.stream(returned_job.name)

## Register Model

In [ ]:
model_name = "iris-sklearn-model"
model_asset = Model(
    path=f"azureml://jobs/{returned_job.name}/outputs/model_dir",
    name=model_name,
    type="custom_model",
    description="Iris sklearn model",
)
registered_model = ml_client.models.create_or_update(model_asset)
print("Registered model:", registered_model.name, "version:", registered_model.version)

## Create Endpoint and Deployment

In [ ]:
inference_env = Environment(
    name="iris-inference-env",
    description="Iris inference env",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    conda_file=str(project_dir / "conda.yml"),
)
inference_env = ml_client.environments.create_or_update(inference_env)

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Iris online endpoint",
    auth_mode="key",
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=registered_model,
    environment=inference_env,
    code_configuration=CodeConfiguration(
        code=str(project_dir),
        scoring_script="score.py",
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("Deployment complete")

## Get Scoring URI and Key

In [ ]:
endpoint_info = ml_client.online_endpoints.get(endpoint_name)
keys = ml_client.online_endpoints.get_keys(endpoint_name)

print("AZURE_ML_SCORING_URI=", endpoint_info.scoring_uri)
print("AZURE_ML_API_KEY=", keys.primary_key)
print("AZURE_ML_DEPLOYMENT_NAME=blue")

## Quick Test

In [ ]:
import requests

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {keys.primary_key}",
    "azureml-model-deployment": "blue",
}
payload = {"data": [5.1, 3.5, 1.4, 0.2]}
r = requests.post(endpoint_info.scoring_uri, headers=headers, json=payload, timeout=30)
print(r.status_code)
print(r.text)

## Connect To Your Website
Set these environment variables in your Node app:
- MODEL_PROVIDER=azureml
- AZURE_ML_SCORING_URI=<value printed above>
- AZURE_ML_API_KEY=<value printed above>
- AZURE_ML_DEPLOYMENT_NAME=blue

In [ ]:
# Optional cleanup to avoid charges
# ml_client.online_endpoints.begin_delete(name=endpoint_name).result()